In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import sys
sys.path.append('../')

import superstats as sup

import jax
print(jax.devices())

## Constants

In [ ]:
NUM_STEPS = 400
NUM_SAMPLES = 2000
NUM_EPOCHS = 100
NUM_ITER_PER_EPOCH = 1000
BATCH_SIZE = 32

## Prior

In [ ]:
joint_prior = sup.prior.JointPrior(
    v = sup.transition.RandomWalk(
            bounds=(0.0, 6.0),
            sigma=sup.prior.Prior(dist="halfnormal", scale=0.1),
            delta=0,
            initial_prior=sup.prior.Prior(dist="normal", loc=-1.5, scale=0.5)
        ),
    a = sup.transition.RandomWalk(
            bounds=(0.0, 4.0),
            sigma=sup.prior.Prior(dist="halfnormal", scale=0.1),
            delta=0,
            initial_prior=sup.prior.Prior(dist="normal", loc=0.0, scale=0.5)
        ),
    tau = sup.transition.RandomWalk(
            bounds=(0.0, 2.0),
            sigma=sup.prior.Prior(dist="halfnormal", scale=0.1),
            delta=0,
            initial_prior=sup.prior.Prior(dist="normal", loc=-1.5, scale=0.5)
        ),
    bias = 0.5
)

In [ ]:
fig = joint_prior.plot_joint_prior(
    num_steps=NUM_STEPS,
    num_trajectories=100,
)

## Generative Model

In [ ]:
ddm = sup.simulation.sample_ddm

generative_model = sup.simulation.GenerativeModel(
    prior=joint_prior,
    model=ddm,
)

In [ ]:
fig = generative_model.plot_push_forward(
    num_sim=10,
    num_steps=NUM_STEPS,
    data_dim=0,
    kind="dist",
    num_cols=5,
)

In [ ]:
fig = generative_model.plot_push_forward(
    num_sim=10,
    num_steps=NUM_STEPS,
    data_dim=1,
    kind="dist",
)

## Workflow

In [ ]:
workflow = sup.workflow.Workflow(
    simulator=generative_model,
    checkpoint_filepath="ddm_benchmark"
)

## Training

In [ ]:
history = workflow.fit_online(
    num_steps=NUM_STEPS,
    epochs=NUM_EPOCHS,
    num_batches_per_epoch=NUM_ITER_PER_EPOCH,
    batch_size=BATCH_SIZE
)

In [ ]:
fig = workflow.plot_history(history)